# Claude on Amazon Bedrock — thinking, tools, and prompt caching

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

The three capabilities that make Claude worth its price on `bedrock-mantle`:
extended/adaptive thinking, tool use with forced schemas, and prompt caching with
a one-hour TTL option.

**Prerequisite:** `01-messages-api-core.ipynb` (auth, the Messages API, the
`content` block trap, `temperature` deprecation).

**Models used:** `claude-opus-5`, `claude-sonnet-5`, `claude-opus-4-8`,
`claude-haiku-4-5` — interleaved, because their thinking support differs.

## Region
This notebook pins `us-east-1`. Model availability differs by Region and changes, so
enumerate `GET /v1/models` for whichever Region you plan to use — `01` §11 shows how.

## Self-contained, but see also
- **Messages API basics** → `01-messages-api-core.ipynb`
- **Agentic tool families (computer use, memory)** →
  `03-agentic-computer-use-and-memory.ipynb`
- **Auth, the three URL paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Workspaces / cost attribution / retention** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs anthropic, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `err` | pulls the human-readable message out of an error body, redacted |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `stream_lines` | raw SSE lines from a streaming endpoint, no SDK |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import converse, err, post, safe_print

REGION = "us-east-1"

OPUS5 = "anthropic.claude-opus-5"
SONNET5 = "anthropic.claude-sonnet-5"
OPUS48 = "anthropic.claude-opus-4-8"
HAIKU45 = "anthropic.claude-haiku-4-5"

PREFIX = "/anthropic/v1"
AV = {"anthropic-version": "2023-06-01"}  # required header on mantle


def claude_text(payload: dict) -> str:
    """TEXT blocks only — reasoning models put a `thinking` block first."""
    return "".join(
        b.get("text", "") for b in payload.get("content", []) if b.get("type") == "text"
    )


def thinking_text(payload: dict) -> str:
    """The thinking blocks' text, which is readable for some thinking forms.

    Whether there is anything to read is keyed on the thinking FORM, not on the
    endpoint. Under `thinking: {"type": "adaptive"}` Claude returns a typed
    `thinking` block whose `thinking` string is "" and whose `signature` carries
    the trace encrypted. Under extended thinking (`"enabled"` plus
    `budget_tokens`) this same endpoint returns the trace as readable text.
    An earlier version of this docstring said "always empty on bedrock-mantle",
    which section 2b disproves. What it cost is reported in
    usage.output_tokens_details.thinking_tokens.
    """
    return "".join(
        b.get("thinking", "")
        for b in payload.get("content", [])
        if b.get("type") == "thinking"
    )


def thinking_signature(payload: dict) -> str:
    """The opaque signature that stands in for the readable trace."""
    return "".join(
        b.get("signature", "")
        for b in payload.get("content", [])
        if b.get("type") == "thinking"
    )


print("endpoint:", f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}")

endpoint: https://bedrock-mantle.us-east-1.api.aws/anthropic/v1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-8",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


anthropic.claude-haiku-4-5             us.anthropic.claude-haiku-4-5-20251001-v1:0 mantle, runtime


anthropic.claude-opus-4-8              us.anthropic.claude-opus-4-8             mantle, runtime
anthropic.claude-opus-5                us.anthropic.claude-opus-5               mantle, runtime
anthropic.claude-sonnet-5              us.anthropic.claude-sonnet-5             mantle, runtime



=> 4/4 of these are on bedrock-runtime; 4 under a different id.
   Every model here is on both endpoints. This notebook shows the
   bedrock-mantle calls; the ids above are what you send to switch.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. Adaptive thinking — and which models have it

`thinking: {"type": "adaptive"}` lets Claude decide how much to think per
request. It is **not** available on every model: `haiku-4-5` rejects it. Probe
before you depend on it.

In [3]:
print(f"{'model':32} {'adaptive thinking':>19}")
print("-" * 54)
for model in (OPUS5, SONNET5, OPUS48, HAIKU45):
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": 64,
            "thinking": {"type": "adaptive"},
            "messages": [{"role": "user", "content": "Reply OK"}],
        },
        region=REGION,
        headers=AV,
    )
    verdict = "supported" if code == 200 else f"{code}"
    print(f"{model:32} {verdict:>19}")
    if code != 200:
        print(f"      {err(data)[:80]}")

model                              adaptive thinking
------------------------------------------------------


anthropic.claude-opus-5                    supported


anthropic.claude-sonnet-5                  supported


anthropic.claude-opus-4-8                  supported
anthropic.claude-haiku-4-5                       400
      adaptive thinking is not supported on this model


## 2. Reading a thinking response — the block, and when its text is readable

With thinking active the `content` array carries a `thinking` block alongside
`text`, which is why `content[0].text` is unsafe.

Whether you can read the reasoning depends on the thinking **form**, not on the
endpoint. Under adaptive thinking the block comes back like this, with the trace
carried in `signature` in encrypted form — the same trade Grok makes with
`reasoning.encrypted_content` (`../11-xai-grok/01` §4c):

```json
{"type": "thinking", "thinking": "", "signature": "CAISiQIKcAgQEAEYAipApexf3L5..."}
```

Under extended thinking (`"enabled"` plus `budget_tokens`) this same
`bedrock-mantle` endpoint returns the trace as readable text instead. Of the models
here only `haiku-4-5` accepts that form, so it is the whole of the evidence for this:
§2b measures which models accept which form and computes which forms produced
readable text, rather than generalising from one to the family.

So the things to read are: whether a `thinking` block is present, how many
`thinking_tokens` it consumed, whether `thinking` itself holds text, and — if you
plan to continue the conversation — the `signature`.

**Which models emit the block is not what you would guess**, either. It depends on
the model *and* on whether you set `thinking` at all, so the cell probes both.

In [4]:
PUZZLE = (
    "A farmer must cross a river with a wolf, a goat and a cabbage. The boat "
    "holds the farmer plus one item. The wolf eats the goat if left alone "
    "together; the goat eats the cabbage. Give the shortest sequence."
)

print(f"{'model':28} {'request':22} {'blocks':26} {'think tok':>10} {'text len':>9}")
print("-" * 100)
for model in (OPUS5, SONNET5, OPUS48):
    for label, extra in (
        ("no thinking param", {}),
        ("thinking: adaptive", {"thinking": {"type": "adaptive"}}),
    ):
        code, data = post(
            f"{PREFIX}/messages",
            {"model": model, "max_tokens": 4000,
             "messages": [{"role": "user", "content": PUZZLE}], **extra},
            region=REGION,
            headers=AV,
        )
        if code != 200:
            print(f"{model:28} {label:22} HTTP {code} {err(data)[:40]}")
            continue
        blocks = [b.get("type") for b in data.get("content", [])]
        details = (data.get("usage") or {}).get("output_tokens_details") or {}
        print(
            f"{model:28} {label:22} {str(blocks):26} "
            f"{details.get('thinking_tokens', 0):>10} {len(claude_text(data)):>9}"
        )

# Show the block itself, so the `thinking` field and the signature are visible.
code, data = post(
    f"{PREFIX}/messages",
    {"model": OPUS5, "max_tokens": 3000,
     "messages": [{"role": "user", "content": PUZZLE}]},
    region=REGION,
    headers=AV,
)
print()
for block in data.get("content", []):
    if block.get("type") == "thinking":
        print("thinking block keys :", sorted(block.keys()))
        trace = block.get("thinking", "")
        # Do not hard-code "always empty": under extended thinking this same
        # field carries readable text (§2b). Describe what this block holds.
        held = f"{len(trace)} chars, READABLE" if trace else "empty -> see signature"
        print(f"  thinking (text)   : {trace[:44]!r}  <- {held}")
        print(f"  signature         : {len(block.get('signature', ''))} chars, opaque")
details = (data.get("usage") or {}).get("output_tokens_details") or {}
print(f"  thinking_tokens   : {details.get('thinking_tokens')}  <- what it cost")
helper_trace = thinking_text(data)
print(f"\nhelper thinking_text() -> {helper_trace[:44]!r} "
      f"({len(helper_trace)} chars)")
print("=== ANSWER ===")
print(claude_text(data)[:400])

model                        request                blocks                      think tok  text len
----------------------------------------------------------------------------------------------------


anthropic.claude-opus-5      no thinking param      ['thinking', 'text']              130      1132


anthropic.claude-opus-5      thinking: adaptive     ['thinking', 'text']              123      1121


anthropic.claude-sonnet-5    no thinking param      ['text']                            0      1141


anthropic.claude-sonnet-5    thinking: adaptive     ['thinking', 'text']              420      1206


anthropic.claude-opus-4-8    no thinking param      ['text']                            0      1090


anthropic.claude-opus-4-8    thinking: adaptive     ['thinking', 'text']               89       660



thinking block keys : ['signature', 'thinking', 'type']
  thinking (text)   : ''  <- empty -> see signature
  signature         : 576 chars, opaque
  thinking_tokens   : 98  <- what it cost

helper thinking_text() -> '' (0 chars)
=== ANSWER ===
## Shortest Solution — 7 Crossings

| # | Move | Left bank (start) | Right bank (goal) |
|---|------|-------------------|-------------------|
| 1 | Farmer takes **goat** over | wolf, cabbage | goat |
| 2 | Farmer returns **alone** | wolf, cabbage, farmer | goat |
| 3 | Farmer takes **wolf** over | cabbage | wolf, goat |
| 4 | Farmer returns **with goat** | cabbage, goat, farmer | wolf |
| 5 | Farm


### 2b. `thinking.type` — four forms, measured per model

Four spellings of `thinking` are in circulation: omitted entirely,
`{"type": "adaptive"}`, the older `{"type": "enabled", "budget_tokens": N}`, and
`{"type": "disabled"}`. Which of them a model accepts is a per-model fact, and so
is whether the thinking trace comes back as readable text, so the cell below
measures both for every model this notebook covers rather than generalising from
one model to the family.

Some of the refusals name the field to send instead and some only say the form is
unsupported, so compare the 400 rows in the table rather than relying on the message
to guide you. `{"type": "enabled", "budget_tokens": N}` is the form a lot of existing
Claude code still sends, and `{"type": "disabled"}` is what gets reached for
to turn thinking off for latency, so it is worth knowing per model which of them the
model you are calling will take.

In [5]:
# Acceptance and trace-readability are both per-model facts, so both are measured
# per model. A single-model probe is what produced the claim this cell replaces:
# "thinking text is never readable on bedrock-mantle", inferred from adaptive-only
# models and false for a model that takes extended thinking.
FORMS = (
    ("omitted", None),
    ("adaptive", {"type": "adaptive"}),
    ("enabled+budget", {"type": "enabled", "budget_tokens": 1024}),
    ("disabled", {"type": "disabled"}),
)
MODELS = (OPUS5, SONNET5, OPUS48, HAIKU45)
PROMPT = "Why is 1/0 undefined? One sentence."

results = {}
print(f"{'model':32} {'thinking form':15} {'HTTP':>5} {'stop':>10} "
      f"{'think tok':>9} {'trace text':>12}")
print("-" * 92)
for model in MODELS:
    for label, thinking in FORMS:
        body = {
            "model": model,
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": PROMPT}],
        }
        if thinking is not None:
            body["thinking"] = thinking
        code, data = post(f"{PREFIX}/messages", body, region=REGION, headers=AV)
        if code != 200:
            results[(model, label)] = (code, 0, 0)
            print(f"{model:32} {label:15} {code:>5} {'-':>10} {'-':>9} {'-':>12}")
            print(f"      {err(data)[:104]}")
            continue
        details = (data.get("usage") or {}).get("output_tokens_details") or {}
        tokens = details.get("thinking_tokens", 0)
        trace = thinking_text(data)
        stop = data.get("stop_reason", "?")
        results[(model, label)] = (code, len(trace), tokens)
        # stop_reason is printed because a truncated answer would otherwise look
        # like a capability result; "max_tokens" here would mean the cap, not the model.
        print(f"{model:32} {label:15} {code:>5} {stop:>10} {tokens:>9} "
              f"{(str(len(trace)) + ' chars') if trace else 'empty':>12}")

# Every verdict below is computed from `results`, so it moves if the service does.
readable = sorted({f for (_, f), (c, n, _) in results.items() if c == 200 and n})
encrypted = sorted({f for (_, f), (c, n, t) in results.items() if c == 200 and not n and t})
print()
if readable and encrypted:
    print(f"=> trace text came back READABLE under: {', '.join(readable)}")
    print(f"   and empty, with the trace in `signature`, under: {', '.join(encrypted)}")
    print("   Same endpoint for both, so readability tracks the form, not the endpoint.")
elif readable:
    print(f"=> readable under {', '.join(readable)}; no form returned an empty trace.")
elif encrypted:
    print("=> every form that produced a thinking block returned empty text this run.")
else:
    print("=> no form produced a thinking block; nothing measured about readability.")

adaptive_ok = {m for m in MODELS if results[(m, "adaptive")][0] == 200}
extended_ok = {m for m in MODELS if results[(m, "enabled+budget")][0] == 200}
disabled_ok = {m for m in MODELS if results[(m, "disabled")][0] == 200}
print()
print(f"   adaptive accepted by {len(adaptive_ok)}/{len(MODELS)}, "
      f"enabled+budget by {len(extended_ok)}/{len(MODELS)}, "
      f"disabled by {len(disabled_ok)}/{len(MODELS)}")
if adaptive_ok and extended_ok and not (adaptive_ok & extended_ok):
    print("   No model here takes both adaptive and enabled+budget, so there is no")
    print("   single way to turn thinking ON across this set: gate the form per model.")
    print("   (Forms accepted by every model above, if any, are listed as 4/4.)")

model                            thinking form    HTTP       stop think tok   trace text
--------------------------------------------------------------------------------------------


anthropic.claude-opus-5          omitted           200   end_turn        27        empty


anthropic.claude-opus-5          adaptive          200   end_turn        61        empty
anthropic.claude-opus-5          enabled+budget    400          -         -            -
      "thinking.type.enabled" is not supported for this model. Use "thinking.type.adaptive" and "output_config


anthropic.claude-opus-5          disabled          200   end_turn         0        empty


anthropic.claude-sonnet-5        omitted           200   end_turn         0        empty


...

anthropic.claude-opus-4-8        adaptive          200   end_turn         0        empty
anthropic.claude-opus-4-8        enabled+budget    400          -         -            -
      "thinking.type.enabled" is not supported for this model. Use "thinking.type.adaptive" and "output_config


anthropic.claude-opus-4-8        disabled          200   end_turn         0        empty


anthropic.claude-haiku-4-5       omitted           200   end_turn         0        empty
anthropic.claude-haiku-4-5       adaptive          400          -         -            -
      adaptive thinking is not supported on this model


anthropic.claude-haiku-4-5       enabled+budget    200   end_turn       160    523 chars


anthropic.claude-haiku-4-5       disabled          200   end_turn         0        empty

=> trace text came back READABLE under: enabled+budget
   and empty, with the trace in `signature`, under: adaptive, omitted
   Same endpoint for both, so readability tracks the form, not the endpoint.

   adaptive accepted by 3/4, enabled+budget by 1/4, disabled by 4/4
   No model here takes both adaptive and enabled+budget, so there is no
   single way to turn thinking ON across this set: gate the form per model.
   (Forms accepted by every model above, if any, are listed as 4/4.)


## 3. Thinking is incompatible with some parameters

When thinking is on, sampling knobs and forced tool use are rejected. That is
documented Anthropic behaviour, and worth seeing rather than discovering.

In [6]:
combinations = [
    ("thinking alone", {"thinking": {"type": "adaptive"}}),
    ("thinking + temperature", {"thinking": {"type": "adaptive"}, "temperature": 0.5}),
    ("thinking + top_p", {"thinking": {"type": "adaptive"}, "top_p": 0.9}),
]
for label, extra in combinations:
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": SONNET5,
            "max_tokens": 200,
            "messages": [{"role": "user", "content": "Reply OK"}],
            **extra,
        },
        region=REGION,
        headers=AV,
    )
    print(f"  {label:26} -> HTTP {code} {'' if code == 200 else err(data)[:70]}")

  thinking alone             -> HTTP 200 
  thinking + temperature     -> HTTP 400 `temperature` may only be set to 1 when thinking is enabled or in adap


  thinking + top_p           -> HTTP 400 `top_p` must be greater than or equal to 0.95 or unset when thinking i


Note that some Claude models reject sampling parameters outright — `01` probes
this and prints the current answer. When a model does its own reasoning, the safe
default is to send no sampling parameters and let it decide.

## 4. Streaming a thinking response

Thinking and answer text arrive as distinct block types, so a client *can* render
them separately. The cell below streams with adaptive thinking, and under that form
the trace is encrypted (§2), so `thinking_delta` events carry no text even though a
`thinking` block opens and closes. It counts both channels so you can see which
events actually arrive. A model that accepts extended thinking (§2b) is where
`thinking_delta` text would be expected instead.

In [7]:
from bedrock import stream_lines

events, thinking_chars, text_chars = {}, 0, 0
print("--- live (thinking dimmed) ---")
for line in stream_lines(
    f"{PREFIX}/messages",
    {
        "model": SONNET5,
        "max_tokens": 2000,
        "stream": True,
        "thinking": {"type": "adaptive"},
        "messages": [
            {"role": "user", "content": "Why is 1/0 undefined? Two sentences."}
        ],
    },
    region=REGION,
    headers=AV,
):
    if not line.startswith("data: "):
        continue
    try:
        event = json.loads(line[6:])
    except json.JSONDecodeError:
        continue
    etype = event.get("type", "")
    events[etype] = events.get(etype, 0) + 1
    if etype == "content_block_delta":
        delta = event.get("delta", {})
        if delta.get("type") == "thinking_delta":
            chunk = delta.get("thinking", "")
            thinking_chars += len(chunk)
            print("\033[2m" + chunk + "\033[0m", end="", flush=True)
        elif delta.get("type") == "text_delta":
            chunk = delta.get("text", "")
            text_chars += len(chunk)
            print(chunk, end="", flush=True)

print(f"\n\nthinking chars: {thinking_chars} | answer chars: {text_chars}")
print("--- event types ---")
for name, count in sorted(events.items(), key=lambda kv: -kv[1]):
    print(f"  {count:4}  {name}")

# Derived: the previous version printed "thinking chars: 0" under a heading
# promising a visible thinking panel, with nothing to explain the zero.
print()
if thinking_chars:
    print("=> thinking_delta carried text on this run.")
else:
    print("=> 0 thinking chars, as expected for adaptive thinking: a thinking block")
    print("   may open and close, but its text is encrypted (§2). Budget for the")
    print("   tokens; do not build a UI that expects to display them. This is a")
    print("   property of the form, not of bedrock-mantle -- see §2b.")

--- live (thinking dimmed) ---


Division

 by zero is undefined because

 division

 is

...

 so

 m

athematicians leave it und

efined instead.



thinking chars: 0 | answer chars: 324
--- event types ---
    25  content_block_delta
     1  message_start
     1  content_block_start
     1  content_block_stop
     1  message_delta
     1  message_stop

=> 0 thinking chars, as expected for adaptive thinking: a thinking block
   may open and close, but its text is encrypted (§2). Budget for the
   tokens; do not build a UI that expects to display them. This is a
   property of the form, not of bedrock-mantle -- see §2b.


## 5. Tool use

The Messages API uses `input_schema` (not `parameters`), and tool results go back
as a `tool_result` block inside a **user** turn.

In [8]:
RATES = {("EUR", "USD"): 1.09, ("USD", "SGD"): 1.34}


def convert(amount: float, source: str, target: str) -> dict:
    rate = RATES.get((source.upper(), target.upper()))
    return {
        "amount": amount,
        "from": source.upper(),
        "to": target.upper(),
        "rate": rate,
        "converted": round(amount * rate, 2) if rate else None,
    }


convert_tool = {
    "name": "convert_currency",
    "description": "Convert an amount between two currencies.",
    "input_schema": {  # note: input_schema, not parameters
        "type": "object",
        "properties": {
            "amount": {"type": "number"},
            "source": {"type": "string"},
            "target": {"type": "string"},
        },
        "required": ["amount", "source", "target"],
    },
}

history = [{"role": "user", "content": "How much is 500 EUR in USD?"}]
code, first = post(
    f"{PREFIX}/messages",
    {"model": SONNET5, "max_tokens": 600, "tools": [convert_tool], "messages": history},
    region=REGION,
    headers=AV,
)
print("stop_reason:", first.get("stop_reason"))
tool_uses = [b for b in first.get("content", []) if b.get("type") == "tool_use"]
print("tool_use blocks:", [(b["name"], b["input"]) for b in tool_uses])

stop_reason: tool_use
tool_use blocks: [('convert_currency', {'amount': 500, 'source': 'EUR', 'target': 'USD'})]


In [9]:
if tool_uses:
    # Echo the assistant turn, then return results in a USER turn.
    history.append({"role": "assistant", "content": first["content"]})
    results = []
    for block in tool_uses:
        results.append(
            {
                "type": "tool_result",
                "tool_use_id": block["id"],  # ties result to request
                "content": json.dumps(convert(**block["input"])),
            }
        )
    history.append({"role": "user", "content": results})

    code, final = post(
        f"{PREFIX}/messages",
        {
            "model": SONNET5,
            "max_tokens": 400,
            "tools": [convert_tool],
            "messages": history,
        },
        region=REGION,
        headers=AV,
    )
    print("final answer:", claude_text(final)[:250])

final answer: 500 EUR converts to **545.00 USD**, based on an exchange rate of 1 EUR = 1.09 USD.


## 6. Forced tools = structured output

Whether Claude's `output_config.format` is accepted here is worth probing rather
than assuming (`01` §7 does), so a
forced tool is the way to get schema-enforced JSON. The result arrives already
parsed as a dict — no string parsing, no trailing-character risk.

In [10]:
ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": {
        "risk_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "concerns": {"type": "array", "items": {"type": "string"}},
        "recommendation": {"type": "string"},
    },
    "required": ["risk_level", "concerns", "recommendation"],
}

emit_tool = {
    "name": "emit_analysis",
    "description": "Return the structured risk analysis.",
    "input_schema": ANALYSIS_SCHEMA,
}


def structured_analysis(text: str, model: str = SONNET5, attempts: int = 3) -> dict:
    """Forced-tool structured output with a retry.

    Forcing a tool is honoured almost always, not always — handle the turn that
    comes back as prose instead of indexing blindly.
    """
    for attempt in range(attempts):
        code, data = post(
            f"{PREFIX}/messages",
            {
                "model": model,
                "max_tokens": 900,
                "tools": [emit_tool],
                "tool_choice": {"type": "tool", "name": "emit_analysis"},
                "messages": [{"role": "user", "content": text}],
            },
            region=REGION,
            headers=AV,
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        blocks = [b for b in data.get("content", []) if b.get("type") == "tool_use"]
        if blocks:
            if attempt:
                print(f"(succeeded on attempt {attempt + 1})")
            return blocks[0]["input"]  # already a dict
        print(f"attempt {attempt + 1}: no tool_use block — retrying")
    raise RuntimeError("model would not emit the forced tool")


print(
    json.dumps(
        structured_analysis(
            "Review this clause: 'The supplier accepts unlimited liability for all "
            "losses.'"
        ),
        indent=2,
    )
)

{
  "risk_level": "high",
  "concerns": [
    "Unlimited liability exposes the supplier to potentially catastrophic financial risk with no cap on damages, regardless of the nature or cause of the loss.",
    "The clause fails to distinguish between direct, indirect, consequential, and incidental damages, meaning the supplier could be liable for speculative or remote losses (e.g., lost profits, business interruption).",
    "There is no carve-out or exclusion for losses caused by the customer's own negligence, misuse, or third-party actions, potentially making the supplier liable even when not at fault.",
    "No liability cap tied to contract value, insurance coverage, or a reasonable multiple thereof, making the risk disproportionate to the value of the engagement.",
    "The clause does not specify whether this liability is subject to a duty to mitigate damages by the counterparty.",
    "Such broad exposure may be uninsurable or may only be partially covered by standard commercial l

## 7. Prompt caching

Mark a cacheable prefix with `cache_control` on a content block. Order matters:
checkpoints are processed **tools → system → messages**, and changing an earlier
section invalidates the later ones.

In [11]:
HANDBOOK = (
    """
PLATFORM HANDBOOK (excerpt)

1. Tiers. Priority for latency-critical customer paths. Standard by default.
   Flex for evaluations and batch-shaped work that tolerates queueing.
2. Retries. Exponential backoff with jitter. 429 and 5xx are transient and must
   be retried. Other 4xx indicate a client defect and must not be retried.
3. Retention. Workloads handling regulated data run with zero data retention and
   replay conversation history client-side.
4. Observability. Only client-error metrics are published; track server errors
   from client-side telemetry.
5. Attribution. Every workload runs under its own tagged project. Untagged usage
   is charged to a shared pool.
"""
    * 8
)  # comfortably over the per-checkpoint token minimum

print(f"handbook ≈ {len(HANDBOOK) // 4} tokens")


def cached_ask(question: str, model: str = SONNET5, ttl: str | None = None) -> dict:
    cache_control = {"type": "ephemeral"}
    if ttl:
        cache_control["ttl"] = ttl
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": 300,
            "system": [
                {"type": "text", "text": HANDBOOK, "cache_control": cache_control}
            ],
            "messages": [{"role": "user", "content": question}],
        },
        region=REGION,
        headers=AV,
    )
    if code != 200:
        raise RuntimeError(f"HTTP {code}: {err(data)}")
    usage = data.get("usage", {})
    return {
        "text": claude_text(data),
        "input": usage.get("input_tokens"),
        "cache_write": usage.get("cache_creation_input_tokens"),
        "cache_read": usage.get("cache_read_input_tokens"),
    }


cold = cached_ask("Which tier suits evaluations? One line.")
print("call 1:", {k: v for k, v in cold.items() if k != "text"})
print("   answer:", cold["text"][:90])

warm = cached_ask("What must untagged usage expect? One line.")
print("\ncall 2:", {k: v for k, v in warm.items() if k != "text"})
print("   answer:", warm["text"][:90])

# Describe what happened rather than labelling the calls cold/warm in advance. The
# cache survives between runs of this notebook, so on any re-run BOTH calls read
# from it and a hardcoded "call 1 (cold)" is simply wrong.
print()
if cold["cache_write"] and warm["cache_read"]:
    print(f"call 1 wrote {cold['cache_write']} tokens, call 2 read {warm['cache_read']}"
          " back — a cold write then a warm read.")
elif warm["cache_read"]:
    print(f"both calls read {warm['cache_read']} tokens from cache: the handbook was")
    print("already warm from an earlier run. Caching persists across processes,")
    print("which is exactly why it pays in production.")
else:
    print("no cache activity — check the prefix clears the per-checkpoint minimum.")

handbook ≈ 1362 tokens


call 1: {'input': 17, 'cache_write': 0, 'cache_read': 1947}
   answer: Flex — it's built for evaluations and batch-shaped work that can tolerate queueing.



call 2: {'input': 16, 'cache_write': 0, 'cache_read': 1947}
   answer: Untagged usage gets charged to a shared pool.

both calls read 1947 tokens from cache: the handbook was
already warm from an earlier run. Caching persists across processes,
which is exactly why it pays in production.


## 8. The one-hour TTL

Claude supports a **1-hour** cache TTL as well as the default 5 minutes. Use it
when follow-up requests may arrive more than 5 minutes apart — a long agent
side-quest, or a user who takes their time replying.

In [12]:
for ttl in (None, "5m", "1h"):
    label = ttl or "default (5m)"
    try:
        result = cached_ask("Name one retry rule. One line.", ttl=ttl)
        print(
            f"  ttl={label:12} -> write={result['cache_write']} "
            f"read={result['cache_read']}"
        )
    except RuntimeError as exc:
        print(f"  ttl={label:12} -> {exc}")

print()
print("All three are accepted. The rows look identical because they all hit the")
print("same warm entry -- a TTL governs how long an unread entry SURVIVES, which a")
print("sequence of back-to-back calls cannot show. The 1h option matters when the")
print("next turn may be more than five minutes away, not for throughput.")

  ttl=default (5m) -> write=0 read=1947


  ttl=5m           -> write=0 read=1947


  ttl=1h           -> write=0 read=1947

All three are accepted. The rows look identical because they all hit the
same warm entry -- a TTL governs how long an unread entry SURVIVES, which a
sequence of back-to-back calls cannot show. The 1h option matters when the
next turn may be more than five minutes away, not for throughput.


### 8b. Mixing TTLs in one request — measured

The constraint on mixing TTLs is a **request-validation** rule, so it is worth
measuring rather than stating: the table below is what the service does with each
ordering, and §8's one-TTL-at-a-time probe above cannot show it.

In [13]:
# This is validation, not a caching effect. These blocks are far below the
# per-checkpoint token minimum, so no cache entry is ever created and the 400 still
# fires -- which is why this probe is cheap and does not reuse HANDBOOK.
#
# Every position pair is measured, not one pair alone, because the rule is about
# order and the prose below names all three positions. The row that catches people
# is (default, 1h): an omitted `ttl` is not "unset" for ordering purposes, so §9's
# own pattern -- tools and system cached with a bare {"type": "ephemeral"} -- is a
# single 1h edit away from this error.
SHORT = "You are a terse assistant."
NOOP_TOOL = {
    "name": "noop",
    "description": "Does nothing; present only to carry a cache checkpoint.",
    "input_schema": {"type": "object", "properties": {}},
}


def cc(ttl: str | None) -> dict:
    """cache_control with an explicit ttl, or the endpoint default when None."""
    return {"type": "ephemeral"} if ttl is None else {"type": "ephemeral", "ttl": ttl}


def build(model: str, spec: dict) -> dict:
    """A request whose cache checkpoints are exactly `spec`: position -> ttl."""
    content = [{"type": "text", "text": "Say OK."}]
    if "messages" in spec:
        content[0]["cache_control"] = cc(spec["messages"])
    body = {"model": model, "max_tokens": 16,
            "messages": [{"role": "user", "content": content}]}
    if "tools" in spec:
        body["tools"] = [{**NOOP_TOOL, "cache_control": cc(spec["tools"])}]
    if "system" in spec:
        body["system"] = [{"type": "text", "text": SHORT,
                           "cache_control": cc(spec["system"])}]
    return body


MINUTES = {None: 5, "5m": 5, "1h": 60}  # the default is 5m, which is the point
PAIRS = (("tools", "system"), ("system", "messages"), ("tools", "messages"))
TTL_PAIRS = ((None, "1h"), ("5m", "1h"), ("1h", "5m"))

rows = []
print(f"{'model':26} {'earlier -> later':22} {'ttls':>12} {'HTTP':>5} {'order err':>10}")
print("-" * 80)
for model in (SONNET5, HAIKU45):
    for earlier, later in PAIRS:
        for e_ttl, l_ttl in TTL_PAIRS:
            code, data = post(
                f"{PREFIX}/messages",
                build(model, {earlier: e_ttl, later: l_ttl}),
                region=REGION,
                headers=AV,
            )
            # Classify on the message, not the status alone: a model that refused 1h
            # outright would also be a 400 and must not be counted as an order error.
            #
            # err() truncates at 160 characters by default, which lands mid-sentence
            # on "processed in the" and drops the order this cell prints below.
            message = "" if code == 200 else err(data, limit=250)
            order_error = code != 200 and "must not come after" in message
            rows.append({"earlier": e_ttl, "later": l_ttl, "code": code,
                         "order_error": order_error, "message": message})
            print(f"{model:26} {earlier + ' -> ' + later:22} "
                  f"{str(e_ttl) + ',' + str(l_ttl):>12} {code:>5} {str(order_error):>10}")

# Verdict computed from the rows rather than asserted: does "a later checkpoint may
# not carry a longer TTL than an earlier one" describe every row?
longer_later = [r for r in rows if MINUTES[r["later"]] > MINUTES[r["earlier"]]]
rest = [r for r in rows if MINUTES[r["later"]] <= MINUTES[r["earlier"]]]
rejected = sum(r["order_error"] for r in longer_later)
accepted = sum(r["code"] == 200 for r in rest)
print()
print(f"later TTL longer than earlier: {len(longer_later):2} rows, {rejected} rejected for order")
print(f"later TTL same or shorter:     {len(rest):2} rows, {accepted} accepted")
print(f"rule holds on every row: {rejected == len(longer_later) and accepted == len(rest)}")

default_rows = [r for r in rows if r["earlier"] is None]
print(f"omitted ttl behaves as 5m in {sum(r['order_error'] for r in default_rows)}"
      f"/{len(default_rows)} rows that pair it with a later 1h")

for r in rows:
    if r["order_error"]:
        print()
        print("the service names the processing order itself:")
        print(f"  {r['message']}")
        break

model                      earlier -> later               ttls  HTTP  order err
--------------------------------------------------------------------------------


anthropic.claude-sonnet-5  tools -> system             None,1h   400       True
anthropic.claude-sonnet-5  tools -> system               5m,1h   400       True


anthropic.claude-sonnet-5  tools -> system               1h,5m   200      False
anthropic.claude-sonnet-5  system -> messages          None,1h   400       True


anthropic.claude-sonnet-5  system -> messages            5m,1h   400       True


anthropic.claude-sonnet-5  system -> messages            1h,5m   200      False
anthropic.claude-sonnet-5  tools -> messages           None,1h   400       True
anthropic.claude-sonnet-5  tools -> messages             5m,1h   400       True


anthropic.claude-sonnet-5  tools -> messages             1h,5m   200      False
anthropic.claude-haiku-4-5 tools -> system             None,1h   400       True
anthropic.claude-haiku-4-5 tools -> system               5m,1h   400       True


anthropic.claude-haiku-4-5 tools -> system               1h,5m   200      False
anthropic.claude-haiku-4-5 system -> messages          None,1h   400       True


anthropic.claude-haiku-4-5 system -> messages            5m,1h   400       True


anthropic.claude-haiku-4-5 system -> messages            1h,5m   200      False
anthropic.claude-haiku-4-5 tools -> messages           None,1h   400       True
anthropic.claude-haiku-4-5 tools -> messages             5m,1h   400       True


anthropic.claude-haiku-4-5 tools -> messages             1h,5m   200      False

later TTL longer than earlier: 12 rows, 12 rejected for order
later TTL same or shorter:      6 rows, 6 accepted
rule holds on every row: True
omitted ttl behaves as 5m in 6/6 rows that pair it with a later 1h

the service names the processing order itself:
  system.0.cache_control.ttl: a ttl='1h' cache_control block must not come after a ttl='5m' cache_control block. Note that blocks are processed in the following order: `tools`, `system`, `messages`.


**Read the table.** The rule the rows describe is that a checkpoint may not carry
a longer TTL than one before it, in the order `tools` → `system` → `messages`, and
that an omitted `ttl` counts as 5m for that comparison rather than as absent. The
service states the order in the error message itself, printed above.

And note cache hits are not charged against your rate limit, so caching buys
throughput headroom too.

## 9. Caching tools and system together

The agentic pattern: tool definitions and instructions are static, the
conversation grows. Cache the static part once.

In [14]:
def agent_turn(question: str, model: str = SONNET5) -> dict:
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": 500,
            # Checkpoint order is tools -> system -> messages.
            "tools": [
                {**convert_tool, "cache_control": {"type": "ephemeral"}},
            ],
            "system": [
                {
                    "type": "text",
                    "text": HANDBOOK,
                    "cache_control": {"type": "ephemeral"},
                }
            ],
            "messages": [{"role": "user", "content": question}],
        },
        region=REGION,
        headers=AV,
    )
    if code != 200:
        raise RuntimeError(f"HTTP {code}: {err(data)}")
    usage = data.get("usage", {})
    return {
        "input": usage.get("input_tokens"),
        "write": usage.get("cache_creation_input_tokens"),
        "read": usage.get("cache_read_input_tokens"),
        "text": claude_text(data),
    }


print(f"{'turn':6} {'input':>8} {'write':>8} {'read':>8}  answer")
print("-" * 84)
for i, question in enumerate(
    [
        "Which tier for a nightly eval job?",
        "Which tier for live chat?",
        "Do we retry a 400?",
    ],
    1,
):
    r = agent_turn(question)
    print(
        f"{i:>6} {r['input']:>8} {r['write'] or 0:>8} {r['read'] or 0:>8}  "
        f"{' '.join(r['text'].split())[:40]!r}"
    )
print("\nTurn 1 writes the prefix; later turns read it.")

turn      input    write     read  answer
------------------------------------------------------------------------------------


     1       86        0     2338  'Flex is the right tier for a nightly eva'


     2       82        0     2338  'For live chat, you should use the **Stan'


     3       82        0     2338  'No — a 400 is a client-defect error, not'

Turn 1 writes the prefix; later turns read it.


## 10. Count tokens before you spend them

`count_tokens` includes system prompts and tool definitions — exactly the parts
people forget when budgeting. It is on both endpoints with different model
coverage; `01-messages-api-core.ipynb` §8 and §8b have the matrix.

In [15]:
for label, body in [
    ("bare prompt", {"messages": [{"role": "user", "content": "Hi"}]}),
    (
        "+ handbook system",
        {"system": HANDBOOK, "messages": [{"role": "user", "content": "Hi"}]},
    ),
    (
        "+ handbook + tool",
        {
            "system": HANDBOOK,
            "tools": [convert_tool],
            "messages": [{"role": "user", "content": "Hi"}],
        },
    ),
]:
    code, data = post(
        f"{PREFIX}/messages/count_tokens",
        {"model": SONNET5, **body},
        region=REGION,
        headers=AV,
    )
    print(f"  {label:20} -> {data.get('input_tokens')} tokens")

  bare prompt          -> 9 tokens
  + handbook system    -> 1953 tokens


  + handbook + tool    -> 2414 tokens


## 11. Compare thinking across models

Same puzzle, different models, measuring what thinking costs.

In [16]:
QUESTION = (
    "Two trains 120km apart approach at 40km/h and 60km/h. A bird flies "
    "between them at 80km/h. How far does the bird fly before they meet?"
)

print(f"{'model':32} {'mode':10} {'in':>6} {'out':>6} {'latency':>9}  answer")
print("-" * 104)
for model in (HAIKU45, OPUS48, SONNET5):
    for mode in ("plain", "adaptive"):
        body = {
            "model": model,
            "max_tokens": 2500,
            "messages": [{"role": "user", "content": QUESTION}],
        }
        if mode == "adaptive":
            body["thinking"] = {"type": "adaptive"}
        started = time.perf_counter()
        code, data = post(f"{PREFIX}/messages", body, region=REGION, headers=AV)
        elapsed = time.perf_counter() - started
        if code != 200:
            print(
                f"{model:32} {mode:10} {'-':>6} {'-':>6} {'-':>9}  "
                f"HTTP {code}: {err(data)[:30]}"
            )
            continue
        usage = data.get("usage", {})
        answer = " ".join(claude_text(data).split())
        print(
            f"{model:32} {mode:10} {usage.get('input_tokens', 0):>6} "
            f"{usage.get('output_tokens', 0):>6} {elapsed:>8.2f}s  {answer[:34]!r}"
        )

model                            mode           in    out   latency  answer
--------------------------------------------------------------------------------------------------------


anthropic.claude-haiku-4-5       plain          49    145     1.32s  '# Train and Bird Problem **Find wh'
anthropic.claude-haiku-4-5       adaptive        -      -         -  HTTP 400: adaptive thinking is not suppo


anthropic.claude-opus-4-8        plain          57    383     5.00s  '# Solving the Bird and Trains Prob'


anthropic.claude-opus-4-8        adaptive       57    453     5.63s  '## Setting Up the Problem The key '


anthropic.claude-sonnet-5        plain          57    552     5.73s  '## Setting Up the Problem **Given '


anthropic.claude-sonnet-5        adaptive       57    545     5.96s  '## Setting Up the Problem **Given '


## 12. Production shape

In [17]:
code, workspace = post(
    "/v1/organization/projects",  # control plane is always /v1
    {
        "name": "claude-thinking-samples",
        "tags": {"Application": "ClaudeThinkingDemo", "Environment": "Demo"},
    },
    region=REGION,
)
workspace_id = workspace.get("id")
safe_print("workspace:", code, workspace_id)


class ClaudeClient:
    """Production shape: caching, optional thinking, forced-tool JSON, attribution."""

    ADAPTIVE_MODELS = (OPUS5, SONNET5, OPUS48)  # haiku-4-5 rejects adaptive

    def __init__(
        self, model=SONNET5, region=REGION, workspace=None, system=None, cache_ttl="1h"
    ):
        self.model, self.region, self.workspace = model, region, workspace
        self.system, self.cache_ttl = system, cache_ttl

    def _headers(self):
        headers = dict(AV)
        if self.workspace:
            headers["anthropic-workspace"] = self.workspace
        return headers

    def _system_blocks(self):
        if not self.system:
            return None
        return [
            {
                "type": "text",
                "text": self.system,
                "cache_control": {"type": "ephemeral", "ttl": self.cache_ttl},
            }
        ]

    def ask(self, question, *, thinking=False, max_tokens=1000, schema=None):
        body = {
            "model": self.model,
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": question}],
        }
        # Never send temperature: deprecated on frontier Claude models.
        if self.system:
            body["system"] = self._system_blocks()
        if thinking and self.model in self.ADAPTIVE_MODELS:
            body["thinking"] = {"type": "adaptive"}
        if schema:
            # output_config.format is rejected on mantle — force a tool instead.
            body["tools"] = [
                {
                    "name": "emit",
                    "description": "Return the result.",
                    "input_schema": schema,
                }
            ]
            body["tool_choice"] = {"type": "tool", "name": "emit"}
        code, data = post(
            f"{PREFIX}/messages", body, region=self.region, headers=self._headers()
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        if schema:
            blocks = [b for b in data.get("content", []) if b.get("type") == "tool_use"]
            if not blocks:
                raise RuntimeError("model did not emit the forced tool")
            return blocks[0]["input"]
        return claude_text(data)


bot = ClaudeClient(system=HANDBOOK, workspace=workspace_id)
print("plain     :", bot.ask("Which tier for batch evals? One line.")[:110])
print(
    "thinking  :", bot.ask("Do we retry a 503? Explain briefly.", thinking=True)[:110]
)
print(
    "structured:",
    json.dumps(
        bot.ask(
            "Assess: 'Unlimited liability for the supplier.'", schema=ANALYSIS_SCHEMA
        )
    ),
)

workspace: 200 proj_kmnajpf5...


plain     : Flex — it's designed for evaluations and batch-shaped work that can tolerate queueing.


thinking  : Yes. Per the handbook's retry policy (§2), **5xx errors are transient and must be retried**, and 503 falls int


structured: {"risk_level": "high", "concerns": ["The statement 'Unlimited liability for the supplier' indicates the supplier (platform provider) bears uncapped financial/legal exposure for any damages arising from the workload or service", "This is a contractual/legal risk unrelated to the technical handbook excerpts provided (tiers, retries, retention, observability, attribution)", "No liability cap increases risk of catastrophic financial exposure from cascading failures, misuse, or third-party claims", "Absence of context on what triggers this liability (e.g., data breach, SLA failure, misuse of regulated data under zero-retention policy) makes the scope of exposure unclear and potentially very broad", "Unlimited liability clauses are atypical in standard platform/vendor agreements and usually signal either a negotiation red flag or a drafting error that should be reviewed by legal counsel"], "recommendation": "Escalate to legal/contracts team immediately for review before proceedin

In [18]:
code, archived = post(
    f"/v1/organization/projects/{workspace_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

archived: 200 archived


## Gotchas — thinking, tools and caching on Claude

| Gotcha | Detail |
|---|---|
| Adaptive thinking | **Not on `haiku-4-5`** — 400. Gate per model |
| Thinking text | Readability tracks the thinking **form**, not the endpoint: empty under `adaptive` (trace encrypted in `signature`), readable under `"enabled"` + `budget_tokens`. Cost is in `thinking_tokens` either way (§2b) |
| `thinking.type` | Four forms (omitted, `adaptive`, `"enabled"` + `budget_tokens`, `disabled`). The two that turn thinking *on* are mutually exclusive across these models — §2b measures which each one takes |
| `content[0]` | Thinking blocks come first — filter by block type |
| Thinking + sampling | `temperature`/`top_p` rejected alongside thinking |
| Tool schema key | `input_schema`, not `parameters` |
| Tool results | Go back in a **user** turn as `tool_result` blocks |
| `output_config.format` | Acceptance varies — probe it (`01` §7); forcing a tool is portable |
| Forced tool | Best-effort; handle the prose turn |
| Cache order | tools → system → messages; earlier edits invalidate later |
| Mixed TTLs | Longer TTL entries must precede shorter ones |
| Cache hits and quota | Not charged against your rate limit |
| `count_tokens` | Counts system + tools. On both endpoints, different coverage — see `01` §8b |

## Next
- `03-agentic-computer-use-and-memory.ipynb` — computer use, memory, compaction
- Other caching model: `../01-openai-gpt/04-prompt-caching-and-cost.ipynb`

## Converse in earnest — the tool loop, provider parameters, and caching

`01-messages-api-core.ipynb` §13 establishes that Claude answers through Converse on
`bedrock-runtime`. That is the easy part. This section does the three things you
actually need there, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.

In [19]:
from bedrock import converse_text, converse_tool_uses, resolve_runtime_id, runtime_client

RUNTIME_ID = "anthropic.claude-sonnet-5"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

# Converse tool shape: toolSpec, and the JSON Schema nests under inputSchema.json.
# This is NOT the OpenAI shape - there is no {"type": "function"} wrapper.
WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

history = [
    {"role": "user", "content": [{"text": "What is the weather in Singapore? Use the tool."}]}
]
first = runtime.converse(
    modelId=resolved,
    messages=history,
    toolConfig={"tools": [WEATHER_TOOL]},
    inferenceConfig={"maxTokens": 500},
)
print("turn 1 stop reason:", first.get("stopReason"))
print("turn 1 blocks     :", [next(iter(b)) for b in first["output"]["message"]["content"]])

uses = converse_tool_uses(first)
if not uses:
    print("no tool call this run - tool_choice defaults to the model's discretion;")
    print("retry, or set toolConfig['toolChoice'] to compel one.")
else:
    use = uses[0]
    print(f"tool call         : {use['name']}({use['input']})")
    # Validate before acting on it. A malformed call still reports tool_use.
    city = str(use["input"].get("city", "")).lower()
    print("arguments valid   :", "yes" if "singapore" in city else f"NO ({use['input']})")

    # Echo the assistant turn back VERBATIM, then answer with a toolResult whose
    # toolUseId matches. Dropping either breaks the loop with a 400.
    history.append(first["output"]["message"])
    history.append(
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": use["toolUseId"],
                        "content": [{"json": {"tempC": 31, "conditions": "humid"}}],
                    }
                }
            ],
        }
    )
    second = runtime.converse(
        modelId=resolved,
        messages=history,
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("turn 2 stop reason:", second.get("stopReason"))
    print("final answer      :", converse_text(second).strip()[:160])


turn 1 stop reason: tool_use
turn 1 blocks     : ['reasoningContent', 'toolUse']
tool call         : get_weather({'city': 'Singapore'})
arguments valid   : yes


turn 2 stop reason: end_turn
final answer      : The current weather in Singapore is **31°C** with **humid** conditions. It's a typical warm, sticky day in the city—stay hydrated if you're heading outside! ☀️💧


In [20]:
# Converse normalises maxTokens, temperature, topP and stopSequences. Anything
# provider-specific goes through additionalModelRequestFields, unvalidated by
# Converse and passed to the provider as-is. That makes it powerful and sharp:
# a key this model does not recognise is a 400, not a silent no-op.
from bedrock import converse_reasoning

PUZZLE = (
    "A bat and ball cost $1.10 together. The bat costs $1.00 more than the ball. "
    "How much is the ball?"
)

for label, extra in [
    ("no extra fields", None),
    ("provider fields", {"thinking": {"type": "adaptive"}, "output_config": {"effort": "high"}}),
]:
    kwargs = {"additionalModelRequestFields": extra} if extra else {}
    try:
        response = runtime.converse(
            modelId=resolved,
            messages=[{"role": "user", "content": [{"text": PUZZLE}]}],
            inferenceConfig={"maxTokens": 900},
            **kwargs,
        )
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__}: {str(exc)[-90:]}")
        continue
    blocks = [next(iter(b)) for b in response["output"]["message"]["content"]]
    trace = converse_reasoning(response)
    answer = converse_text(response).strip().replace("\n", " ")
    print(f"{label:<16} out={response['usage']['outputTokens']:>4} blocks={blocks}")
    print(f"{'':<16} reasoning={len(trace)} chars | {answer[:70]}")

print()
print("Note whether a reasoningContent block appears above. Some models return the")
print("trace as a typed block on Converse and some do not, so read the blocks")
print("rather than assuming - and never index content[0].")


no extra fields  out= 257 blocks=['text']
                 reasoning=0 chars | # Bat and Ball Problem  **Setting up the equations:** - Let ball = $x$


provider fields  out= 365 blocks=['text']
                 reasoning=0 chars | # Bat and Ball Problem  **The ball costs $0.05 (5 cents)**  ## The Mat

Note whether a reasoningContent block appears above. Some models return the
trace as a typed block on Converse and some do not, so read the blocks
rather than assuming - and never index content[0].


In [21]:
# cachePoint marks a prefix as cacheable. Everything BEFORE the marker is cached;
# the marker goes last in the block list it applies to. Watch the usage fields:
# the first call writes, the second reads.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)


def cached_call():
    return runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )


print(f"{'call':<6} {'write':>8} {'read':>8}  total")
print("-" * 40)
for n in (1, 2):
    usage = cached_call()["usage"]
    print(
        f"{n:<6} {str(usage.get('cacheWriteInputTokens')):>8} "
        f"{str(usage.get('cacheReadInputTokens')):>8}  {usage['totalTokens']}"
    )

print()
print("Call 1 writes the prefix, call 2 reads it back. Cached input is billed at a")
print("lower rate than fresh input, so a long stable system prompt reused across")
print("many calls is where this pays. Check the pricing page for the current ratio.")


call      write     read  total
----------------------------------------


1          None     4332  4370


2          None     4332  4370

Call 1 writes the prefix, call 2 reads it back. Cached input is billed at a
lower rate than fresh input, so a long stable system prompt reused across
many calls is where this pays. Check the pricing page for the current ratio.


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.
